# BraTS-GLI: enhancing-tumour post-processing (GPU)

Companion notebook for [MHKhan50/brats-glioma-segmentation](https://github.com/MHKhan50/brats-glioma-segmentation).

**Add Input → Your Work:** (1) the output of **"BraTS-GLI Download and Preprocess (CPU)"** and (2) the output of
**"BraTS-GLI 3D U-Net vs SwinUNETR (GPU)"** (the version with all six trained models).

Kaggle settings: **GPU T4 x2**, **Internet on**, then *Save Version → Save & Run All*. No training: the six trained models
are only used for inference on the validation and test cases (about 4 to 5 hours).

The rule (Isensee et al., 2018): if the predicted enhancing tumour has fewer than T voxels, it is set to empty. T is chosen
on the **validation** cases only, for each model's seed ensemble, and the test set is scored once.

In [ ]:
import os, subprocess, sys
os.chdir("/kaggle/working")
if not os.path.exists("brats-glioma-segmentation"):
    subprocess.run(["git", "clone", "-q", "https://github.com/MHKhan50/brats-glioma-segmentation.git"], check=True)
os.chdir("brats-glioma-segmentation")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train,dev]"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
def bs(*args):
    """Run one CLI step, stream its output and stop with the error text if it fails."""
    proc = subprocess.Popen([sys.executable, "-m", "brats_seg.cli", *args], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in proc.stdout:
        print(line, end="", flush=True); lines.append(line)
    if proc.wait():
        raise RuntimeError("step failed: " + " ".join(args) + "\n" + "".join(lines[-30:]))

In [ ]:
import glob, shutil, json
PREP = os.path.dirname(sorted(glob.glob("/kaggle/input/**/prep/cases.json", recursive=True))[0])
RUNS, REPORTS = "/kaggle/working/runs", "/kaggle/working/reports"
os.makedirs(RUNS, exist_ok=True)
for f in sum((glob.glob(f"/kaggle/input/**/runs/*{s}", recursive=True) for s in ("_best.pt", "_last.pt", "_meta.json")), []):
    if not os.path.exists(os.path.join(RUNS, os.path.basename(f))):
        shutil.copy(f, RUNS); print("restored", os.path.basename(f))
cases = json.load(open(PREP + "/cases.json"))
print(PREP, {s: sum(c["split"] == s for c in cases) for s in ("train", "val", "test")})

In [ ]:
missing = [f"{m}_s{s}_best.pt" for m in ("unet", "swinunetr") for s in (0, 1, 2) if not os.path.exists(os.path.join(RUNS, f"{m}_s{s}_best.pt"))]
assert not missing, f"Attach the training notebook's output; missing: {missing}"
bs("postprocess", "--data", PREP, "--runs", RUNS, "--out", "/tmp/smoke_pp", "--limit", "2")  # quick check on 2 cases

## Validation and test

In [ ]:
bs("postprocess", "--data", PREP, "--runs", RUNS, "--out", REPORTS)
shutil.make_archive("/kaggle/working/brats_postprocess", "zip", REPORTS)
print(open(REPORTS + "/postprocess_summary.json").read())